# moe-sae discovery — bucketless streaming on a Colab A100-40

No GCS, no Drive: activations stream through a rotating on-disk window while **all 14 discovery runs co-train on one GPU** (~5–7 h, ~70 compute units). Works from browser Colab or the VS Code Colab extension.

**Accepted risk (by design):** the stream is not resumable — a session death mid-run means rerunning the run phase from the start. Phases 1–2 (stats, retained eval capture) do checkpoint and resume. Keep the session alive for phase 3.

Artifacts (~25 GB: runs/, dumps/, reports/) are zipped at the end for download.

In [ ]:
gpu = !nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
print(gpu)
assert any("A100" in g for g in gpu), "Runtime > Change runtime type > A100 GPU"
!df -h / | tail -1  # want ~200+ GB free local disk

In [ ]:
import os
token = os.environ.get("GITHUB_TOKEN")
if not token:
    try:
        from google.colab import userdata
        token = userdata.get("GITHUB_TOKEN")
    except Exception:
        from getpass import getpass
        token = getpass("GitHub PAT (repo read): ")
if not os.path.exists("moe-sae"):
    !git clone -q https://{token}@github.com/andrewbistras/moe-sae.git
%cd moe-sae
!git pull -q
!pip -q install -e . --no-deps
!pip -q install "transformers==4.38.2" timm einops sentencepiece safetensors datasets accelerate pyyaml
!python -c "import torch, moe_sae; print('torch', torch.__version__, 'cuda:', torch.cuda.is_available())"

In [ ]:
# Phase 1 — prefix statistics pass (spec §5 'initial statistics pass'):
# 2,000 train-bucket examples → μ_e, μ_route, router baselines; temp deleted.
import os.path
if not os.path.exists("stats/discovery_prefix/stats.safetensors"):
    !python scripts/run_capture.py --out /tmp/prefix_cap --examples 2000 \
        --visual-cap 512 --seed 7 --device cuda --buckets 0-79
    !python scripts/run_stats.py --capture /tmp/prefix_cap --out stats/discovery_prefix
    !rm -rf /tmp/prefix_cap
!python -c "from safetensors.torch import load_file; s=load_file('stats/discovery_prefix/stats.safetensors'); print({k: tuple(v.shape) for k,v in s.items()})"

In [ ]:
# Phase 2 — retained dev/eval capture (buckets 80–99; feeds §13–§17 after
# training). ~6k examples ≈ 48 GB on local disk. Resumable (--resume).
import os.path
resume = "--resume" if os.path.exists("captures/discovery_eval/progress.json") else ""
!python scripts/run_capture.py --out captures/discovery_eval --examples 30000 \
    --visual-cap 512 --seed 7 --device cuda --buckets 80-99 {resume}
# (30k consumed stream examples ≈ 6k kept — the bucket filter skips 80%)

In [ ]:
# Phase 3 — THE RUN: streaming co-training of all 14 discovery conditions.
# ~5–7 h. Keep this session alive (not resumable by design — accepted risk).
!python scripts/run_stream_discovery.py \
    --configs 'configs/discovery_*.yaml' --stats stats/discovery_prefix \
    --examples 96000 --window-batches 800 --device cuda

In [ ]:
# Phase 4 — reconstruction eval + dev dumps + the BINDING §20.5 gate.
for run in ("discovery_A_seed1", "discovery_C_seed1"):
    !python scripts/run_eval.py --run runs/{run} --capture captures/discovery_eval \
        --split dev --dump dumps/{run}/dev --device cuda
!python -c "
from moe_sae.eval.sanity_gate import compare_sanity
v = compare_sanity('runs/discovery_A_seed1', 'runs/discovery_C_seed1',
                   'reports/sanity_gate_discovery.json')
print('DISCOVERY GATE (spec §20.5):', 'PASS' if v['passed'] else 'FAIL',
      '| A/C =', round(v['ratio_a_over_c'], 4))"


In [ ]:
# Phase 5 — package everything that must survive the VM.
!tar czf discovery_artifacts.tgz runs dumps reports stats
!ls -lh discovery_artifacts.tgz
print("Download via the file browser, or files.download('discovery_artifacts.tgz')")